# FASTA2GO - validação da etapa de extração

Pipeline completa, etapa por etapa: ingestão do FASTA -> características -> UniProt -> GO -> resumo.

**Como executar.** Abra este notebook no VS Code com o kernel do `.venv` do projeto. A Parte A
usa um banco descartável construído a partir de `tests/data/sample.fasta` (4 proteínas, incluindo
registros defeituosos) e chama a API real do UniProt. A Parte B inspeciona o banco produzido
pelo CLI com o Swiss-Prot real:

```bash
uv run python main.py run --limit 2000     # cria db/fasta2go.duckdb
```

As células usam `assert`: se algo estiver errado, a execução para.

## Preparação

In [ ]:
from pathlib import Path

import duckdb

from fasta2go import db, features, pipeline

SAMPLE = Path("tests/data/sample.fasta")
SCRATCH = Path("db/validation.duckdb")
REAL = Path("db/fasta2go.duckdb")
SCRATCH.unlink(missing_ok=True)      # a Parte A sempre começa de um banco vazio

conn = db.connect(SCRATCH)
db.init_schema(conn)


def show(sql: str, params=()) -> None:
    """Imprime o resultado de uma consulta como tabela simples."""
    result = conn.execute(sql, list(params))
    print(" | ".join(column[0] for column in result.description))
    for row in result.fetchall():
        print(" | ".join("" if value is None else str(value) for value in row))

## Etapa 1 - Ingestão do FASTA

Cada registro é lido em streaming. Cabeçalho original, identificador, accession (quando o
cabeçalho é UniProt) e sequência são preservados. Registros impossíveis de armazenar são
rejeitados e registrados em `ingestion_issues`; resíduos ambíguos ficam na sequência, mas são
sinalizados.

In [ ]:
ingested = pipeline.ingest(conn, SAMPLE)
print(ingested)

assert ingested["inserted"] == 4
assert ingested["rejected"] == 2      # sequência vazia e cabeçalho sem identificador
assert ingested["duplicates"] == 1    # P69905 aparece duas vezes

show("SELECT fasta_id, uniprot_accession, sequence_length FROM proteins ORDER BY fasta_id")
print()
show("SELECT line_number, fasta_id, issue, detail FROM ingestion_issues ORDER BY line_number")

## Etapa 2 - Características da sequência

Comprimento canônico, composição de aminoácidos (AAC), composição de dipeptídeos (DPC) e
propriedades físico-químicas. O denominador é sempre o número de resíduos canônicos: `X`, `B`,
`Z`, `U`, `O` e `*` são excluídos e contados em `noncanonical_residues`.

In [ ]:
computed = pipeline.compute_features(conn)
print(computed)
assert computed["computed"] == 4

show(
    """
    SELECT p.fasta_id, p.sequence_length, f.canonical_length, f.noncanonical_residues,
           round(f.molecular_weight, 1) AS mw, round(f.isoelectric_point, 2) AS pi,
           round(f.gravy, 3) AS gravy, round(f.sequence_entropy, 3) AS entropy
    FROM proteins p JOIN sequence_features f USING (source_file, fasta_id)
    ORDER BY p.fasta_id
    """
)

# Os denominadores são verificáveis: toda linha de composição soma 100.
for table, columns in (
    ("amino_acid_composition", features.AAC_COLUMNS),
    ("dipeptide_composition", features.DPC_COLUMNS),
):
    expression = " + ".join(f'"{column}"' for column in columns)
    outside = conn.execute(
        f"SELECT count(*) FROM {table} WHERE abs(({expression}) - 100) > 1e-6"
    ).fetchone()[0]
    assert outside == 0, table
print("AAC e DPC: todas as linhas somam 100%")
print("colunas de DPC:", len(features.DPC_COLUMNS))

## Etapa 3 - Enriquecimento pela API do UniProt

Uma requisição por lote de accessions (200 por padrão), com validação prévia dos identificadores
e retry em 429/5xx. Cada lote é gravado em uma transação; uma falha de rede aborta sem registrar
nada como "não encontrado".

In [ ]:
enriched = pipeline.enrich(conn)
print(enriched)
assert enriched["queried"] == 3
assert enriched["matched"] == 3

show(
    """
    SELECT fasta_id, uniprot_id, protein_name, organism, taxon_id,
           protein_existence, source_length, status
    FROM uniprot_annotations ORDER BY fasta_id
    """
)

## Etapa 4 - Anotações de Gene Ontology

As anotações vêm de `uniProtKBCrossReferences` (database `GO`): identificador, aspecto (prefixo
`P`/`C`/`F` do `GoTerm`), nome, código de evidência e quem a atribuiu (`GoEvidenceType`), além
da proveniência (`evidences`, quando existe). Nenhum ancestral é propagado nesta etapa.

In [ ]:
show(
    """
    SELECT aspect, count(*) AS annotations, count(DISTINCT go_id) AS terms,
           count(*) FILTER (WHERE provenance IS NOT NULL) AS with_provenance
    FROM protein_go_annotations GROUP BY aspect ORDER BY aspect
    """
)
print()
show(
    """
    SELECT go_id, aspect, go_name, evidence_code, evidence_source, provenance
    FROM protein_go_annotations WHERE fasta_id = 'sp|P69905|HBA_HUMAN'
    ORDER BY aspect, go_id LIMIT 8
    """
)

print("anotações por proteína:", conn.execute(
    "SELECT round(avg(n), 2) FROM (SELECT count(*) AS n FROM protein_go_annotations "
    "GROUP BY source_file, fasta_id)"
).fetchone()[0])

## Etapa 5 - Idempotência

Repetir as três etapas não insere, não recalcula e não reconsulta nada.

In [ ]:
before = conn.execute(
    "SELECT (SELECT count(*) FROM proteins), (SELECT count(*) FROM sequence_features), "
    "(SELECT count(*) FROM protein_go_annotations)"
).fetchone()

print(pipeline.ingest(conn, SAMPLE))
print(pipeline.compute_features(conn))
print(pipeline.enrich(conn))

after = conn.execute(
    "SELECT (SELECT count(*) FROM proteins), (SELECT count(*) FROM sequence_features), "
    "(SELECT count(*) FROM protein_go_annotations)"
).fetchone()
assert before == after
print("contagens inalteradas:", after)

## Etapa 6 - Retomada após interrupção

Apagar as anotações de uma proteína simula uma interrupção no meio do enriquecimento. A próxima
execução reconsulta **apenas** o que falta, sem reimportar nem recalcular as características.

In [ ]:
conn.execute("DELETE FROM protein_go_annotations WHERE fasta_id = 'sp|P69905|HBA_HUMAN'")
conn.execute("DELETE FROM uniprot_annotations WHERE fasta_id = 'sp|P69905|HBA_HUMAN'")

resumed = pipeline.enrich(conn)
print(resumed)
assert resumed["queried"] == 1
assert resumed["matched"] == 1
assert conn.execute(
    "SELECT count(*) FROM protein_go_annotations WHERE fasta_id = 'sp|P69905|HBA_HUMAN'"
).fetchone()[0] > 0
print("anotações restauradas")

## Etapa 7 - Resumo do banco

O mesmo resumo impresso pelo comando `status`.

In [ ]:
summary = pipeline.database_summary(conn)
for key, value in summary.items():
    print(f"{key:>22}: {value}")

assert summary["orphans"] == 0
assert summary["proteins"] == summary["with_features"] == summary["with_composition"]

## Parte B - Swiss-Prot real (somente leitura)

Banco construído com `uv run python main.py run --limit 2000`, executado duas vezes para
incluir 4.000 proteínas. Esta célula não altera nada.

In [ ]:
if REAL.exists():
    real = duckdb.connect(str(REAL), read_only=True)
    counts = real.execute(
        "SELECT (SELECT count(*) FROM proteins) AS proteins, "
        "(SELECT count(*) FROM protein_go_annotations) AS go_rows, "
        "(SELECT count(DISTINCT (source_file || fasta_id)) FROM protein_go_annotations) "
        "AS proteins_with_go"
    ).fetchone()
    print("proteínas, linhas GO, proteínas com GO:", counts)

    print()
    print("termos de função molecular mais frequentes nesta amostra:")
    for row in real.execute(
        """
        SELECT go_id, any_value(go_name) AS term, count(*) AS proteins
        FROM protein_go_annotations WHERE aspect = 'molecular_function'
        GROUP BY go_id ORDER BY proteins DESC LIMIT 5
        """
    ).fetchall():
        print("  ", row)
    real.close()
else:
    print("db/fasta2go.duckdb não existe: rode `uv run python main.py run --limit 2000`.")

conn.close()